# Hermes Agent - Jupyter Notebook

This notebook allows you to interact with Hermes, the NousResearch AI agent, directly from Jupyter.

## ⚠️ Supabase RLS Setup Required!

Before using this, you must disable RLS on your storage bucket:

1. Go to Supabase Dashboard → **Storage**
2. Click on your bucket (`manini`)
3. Go to **Policies** tab
4. Delete ALL existing policies
5. Click **Add policy** → **Create a public policy**
6. Name it `public_access`
7. For both **SELECT** and **INSERT/UPDATE/DELETE**, use:
   `true`

**Or run this SQL in Supabase SQL Editor:**
```sql
DROP POLICY IF EXISTS "Public Access" ON storage.objects;
CREATE POLICY "Public Access" ON storage.objects
FOR ALL USING (bucket_id = 'manini') WITH CHECK (bucket_id = 'manini');
```

In [ ]:
# Install Supabase client
!pip install --break-system-packages supabase

In [ ]:
# Setup Supabase sync for notebook persistence
import os
from supabase import create_client, Client

# Get credentials from environment
SUPABASE_URL = os.environ.get('SUPABASE_URL', 'https://opdpexsytsaldlworztz.supabase.co')
SUPABASE_KEY = os.environ.get('SUPABASE_KEY', '')  # anon key
BUCKET_NAME = os.environ.get('SUPABASE_BUCKET', 'manini')

NOTEBOOK_DIR = '/data/notebooks'

# Initialize Supabase client
if SUPABASE_URL and SUPABASE_KEY:
    supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    print("✓ Connected to Supabase!")
    print(f"  Bucket: {BUCKET_NAME}")
else:
    supabase = None
    print("⚠️  Set SUPABASE_KEY environment variable")

os.makedirs(NOTEBOOK_DIR, exist_ok=True)
os.chdir(NOTEBOOK_DIR)
print(f"Working directory: {os.getcwd()}")

## 📤 Download Notebooks from Supabase

In [ ]:
def pull_notebooks():
    """Download notebooks from Supabase Storage."""
    if not supabase:
        print("Supabase not configured")
        return
    
    try:
        files = supabase.storage.from_(BUCKET_NAME).list()
        
        if not files:
            print("No notebooks found in storage")
            return
        
        for file in files:
            name = file.get('name', '')
            if name.endswith('.ipynb'):
                print(f"Downloading {name}...")
                data = supabase.storage.from_(BUCKET_NAME).download(name)
                with open(name, 'wb') as f:
                    f.write(data)
        
        print(f"\n✓ Done! Files: {os.listdir('.')}")
        
    except Exception as e:
        print(f"Error: {e}")

pull_notebooks()

## ⬆️ Upload Notebooks to Supabase

In [ ]:
def save_and_sync():
    """Upload notebooks to Supabase Storage."""
    if not supabase:
        print("Supabase not configured")
        return
    
    uploaded = 0
    
    for filename in os.listdir('.'):
        if filename.endswith('.ipynb'):
            print(f"Uploading {filename}...")
            try:
                # Try upload first
                with open(filename, 'rb') as f:
                    supabase.storage.from_(BUCKET_NAME).upload(
                        filename,
                        f.read(),
                        {"contentType": "application/json"}
                    )
                uploaded += 1
                print(f"  ✓ Uploaded")
            except Exception as e:
                # If upload fails (file exists), try update
                try:
                    with open(filename, 'rb') as f:
                        supabase.storage.from_(BUCKET_NAME).update(
                            filename,
                            f.read(),
                            {"contentType": "application/json"}
                        )
                    uploaded += 1
                    print(f"  ✓ Updated")
                except Exception as e2:
                    print(f"  ✗ Error: {e2}")
    
    print(f"\n✓ {uploaded} notebooks synced to Supabase")

save_and_sync()

---

## 🚀 Install Hermes Agent

Run this to install Hermes (one-time):

In [ ]:
!pip install --break-system-packages git+https://github.com/NousResearch/hermes-agent.git

## Initialize Hermes Agent

In [ ]:
from run_agent import AIAgent

# Set your API key:
# os.environ['OPENAI_API_KEY'] = 'your-key'

agent = AIAgent(
    model="openai/gpt-4o",
    quiet_mode=True,
)

print("Hermes Agent initialized!")

## Chat with Hermes

In [ ]:
response = agent.chat("Hello!")
print(response)